# Partitioned Data Lake Architecture: Designing Directory Layouts

**Module**: 08_data_sources_and_formats  
**Notebook**: 03_partitioned_data_lake.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Designing partition directories (`year=YYYY/month=MM/`), directory partitioning strategies, and partition pruning.
- **Why does this topic exist?**: Directory partitioning enables queries to skip 90%+ of files during scanning.
- **What problem does it solve?**: Creating too many partitions (over-partitioning) or partitioning on high-cardinality keys.
- **Where is it used in real-world Data Engineering?**: Enterprise data lake storage layout design.

---

## 2. Data Lake Partition Directory Hierarchy: `warehouse/orders/year=2025/month=09/data.parquet`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Data_Lake_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")
out_lake = "/opt/spark-data/warehouse/lake_orders"

# Add year and month partition columns
lake_df = orders.withColumn("year", F.year("order_ts")).withColumn("month", F.month("order_ts"))
lake_df.write.mode("overwrite").partitionBy("year", "month").parquet(out_lake)
print("Data Lake partitioned layout created successfully!")

---

## Experiment: Partition Pruning Verification

- **1. What are we testing?**: Filter on year=2025 and month=9 and verify that only 1 directory is scanned.
- **2. What do we expect?**: PartitionFilters skips all other month directories.

In [ ]:
read_lake = spark.read.parquet(out_lake)
read_lake.filter("year = 2025 AND month = 9").explain("formatted")

**3. What happened?**: PartitionFilters pushed to FileScan; irrelevant directories were ignored.  
**4. Why did it happen?**: Spark pruned unneeded directories before listing files.  
**5. Practical Engineering Lesson**: Partition by query filter columns to accelerate analytical scans.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Directory Tree
**Problem Statement**: List generated directories under out_lake.  
**Input / Context**: out_lake

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Filter Single Month
**Problem Statement**: Count orders in month 11.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Max Records Per File
**Problem Statement**: Write with maxRecordsPerFile setting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Directory Tree):
```python
print([d for d in os.listdir(out_lake) if d.startswith('year')])
```

#### Solution 2 (Filter Single Month):
```python
print('Count:', read_lake.filter('month = 11').count())
```

#### Solution 3 (Max Records Per File):
```python
lake_df.write.mode('overwrite').option('maxRecordsPerFile', 5000).parquet('/opt/spark-data/warehouse/lake_chunked')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Partition Pruning**: Skips entire directories at the storage layer.
- **Cardinality Rule**: Use low/medium cardinality columns (year, month, region).

### Essential Technical Terms
- `partitionBy`: Writes data into nested subdirectories.
- `PartitionFilters`: Physical operator pruning directories.

### Common Engineering Mistakes
- **Partitioning by timestamp**: Creates millions of empty directories, destroying file system performance.

### Interview Questions on This Topic
- **Q: What is partition pruning?**
  - *A*: Spark reading only matching partition directory folders, ignoring all other folders entirely.

### What to Study Next
Open 04_schema_evolution.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')